# Parse Constitutions into an OHCO Corpus

This notebook builds the normalized constitution corpus from the plaintext files in `individual_constitutions/` and converts them into a flat token-level table using an OHCO-style hierarchy:

`constitution -> article -> clause -> token`

The resulting table is written to `corpus.csv` and can be used as the starting point for later DS 5001 tables such as `LIB`, `CORPUS`, `VOCAB`, `BOW`, and model inputs.


## Inputs and Outputs

**Input folder:** `individual_constitutions/`

Each input file is a plaintext `.txt` file. The first line usually contains a document label such as `Afghanistan 2004`, which the parser uses to identify the constitution, but the exported corpus keeps only country metadata plus normalized text-structural columns.

**Output file:** `corpus.csv`

The output CSV has one row per token and includes the following columns:

- `country`
- `article_n`
- `clause_n`
- `token_n`, `token_str`
- `pos`, `term_str`, `pos_group`


## Constitution Structures Identified

The constitutions use several related structures rather than one perfectly uniform format. The parser identifies a document-level constitution from the file itself and reads the first line mainly to recognize the document and recover the country name. A header like `Afghanistan 2004` helps identify the source document, but the year is not kept in the normalized corpus table because it is document metadata rather than part of the token sequence.

Many constitutions include a preamble before the numbered legal provisions. A preamble usually states the historical, political, religious, or philosophical basis for the constitution. The parser recognizes `Preamble` as a special article-like unit so that its words are kept in the corpus instead of being dropped.

The parser also detects larger organizational divisions, including chapters, parts, titles, and Roman-numeral sections. These divisions help the parser understand local document structure, but they are not exported as corpus columns in the final normalized table because they are not used consistently across all constitutions.

The most important middle-level structure is the article-like unit. Most files use headings such as `Article 1` or `Art. 5`, but some constitutions use `Section`, `Amendment`, or bare numbered headings instead. The parser normalizes all of these article-like structures into the same article level so that documents with different naming conventions can still be compared.

Below articles, the parser treats paragraph-like blocks as clauses. This is where most substantive legal language appears. Some clauses contain smaller internal lists or subclauses marked with numbers, letters, or Roman numerals. The current parser keeps those as clause-level text rather than splitting them into another structural layer. Finally, each clause is split into whitespace-delimited word tokens, producing the token-level rows used for later corpus, vocabulary, and model tables.

For analysis, these source structures are normalized into the OHCO model `Constitution -> Article -> Clause -> Token`, represented in the exported corpus as `country | article_n | clause_n | token_n`.


## Why Larger Divisions Are Metadata

Larger divisions such as chapters, parts, titles, and Roman-numeral sections are useful for interpreting individual constitutions, but they are not included as required levels in the final normalized corpus.

This choice was made because the source documents do not all use the same larger structure. Some constitutions use chapters, some use parts, some use titles, some use Roman-numeral sections, and some have little or no larger division above the article or section level. If every document were forced into a `Constitution -> Chapter -> Article -> Clause -> Token` hierarchy, many documents would have missing, uneven, or misleading chapter-level values. The article-like level is more stable across the corpus, so it becomes the main comparable unit.

The main strength of this approach is consistency. It makes it easier to compare constitutions across countries because every document is normalized into the same core structure: `Constitution -> Article -> Clause -> Token`. It also preserves the most analytically useful units for text analysis: articles as legal units, clauses as paragraph-level provisions, and tokens as word-level observations.

The tradeoff is simplification. Some constitutions use chapters, parts, or titles in meaningful ways, and those labels may describe important themes such as fundamental rights, executive power, courts, emergency powers, or amendment procedures. Because these larger divisions are not part of the exported normalized corpus, analyses based only on the final OHCO columns will emphasize article and clause structure rather than the full document hierarchy.


## Imports

The parser uses Python's standard library, pandas for tables, and NLTK for token-level linguistic annotation. `re` handles structural pattern matching, `Path` handles file paths, pandas stores the table, and NLTK provides POS tags.


In [1]:
import re
import os
import pandas as pd
import nltk
from pathlib import Path

# Download the NLTK resources used for token tagging in this notebook.
for package in ['punkt', 'punkt_tab', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(package, quiet=True)


## Heading Patterns

The source files are fairly consistent, but constitutions do not all use the same hierarchy. Some use `Chapter -> Article`, some use `Article -> Section`, and some use numbered headings such as `1. Name and territory`.

These regular expressions identify the common structural headings and small artifacts in the source text, especially the repeated `Share` lines inherited from web formatting.


In [2]:
RE_CHAPTER = re.compile(
    r'^(Chapter|CHAPTER|Part|PART|Title|TITLE)\s+([IVXivx\d]+)[.):\s]*(.*)',
    re.IGNORECASE
)

RE_ROMAN_SECTION = re.compile(
    r'^([IVX]+)\.\s+(.+)$'           # Match Roman-numeral section headers.
)

RE_ARTICLE = re.compile(
    r'^(Article|Art\.?)\s+(\w+)[.):\s]*(.*)',
    re.IGNORECASE
)

RE_SECTION = re.compile(
    r'^Section\s+(\w+)[.):\s]*(.*)',  # Match section headings used below articles.
    re.IGNORECASE
)

RE_AMENDMENT = re.compile(
    r'^(Amendment)\s+(\w+)[.):\s]*(.*)',
    re.IGNORECASE
)

RE_NUMBERED_SECTION = re.compile(
    r'^(\d+[A-Z]?)\.\s+([A-Z].{3,})$'   # Match bare numbered article-style headings.
)

RE_SUBCLAUSE = re.compile(
    r'^(\d+|[a-z]|[IVX]+)\.\s+\S'       # Match numbered or lettered sub-clauses.
)

RE_PREAMBLE = re.compile(r'^Preamble$', re.IGNORECASE)

RE_SHARE = re.compile(r'^Share$')

RE_DOC_HEADER = re.compile(
    r'^(.+?)\s+(\d{4})(?:\s+\(.*\))?$'
)


## Document Structure Detection

Before parsing a file, the notebook performs a quick scan to infer the document's dominant structure. This matters because the word `Article` may be a top-level unit in one constitution but a nested unit in another.


In [3]:
def _has_chapters(lines: list[str]) -> bool:
    """Return True if the file uses an explicit Chapter/Part/Title above Articles."""
    for line in lines:
        if RE_CHAPTER.match(line) or RE_ROMAN_SECTION.match(line):
            return True
    return False

def _uses_numbered_sections(lines: list[str]) -> bool:
    """Return True if primary article units are bare numbered sections (India/Canada)."""
    article_count = sum(1 for l in lines if RE_ARTICLE.match(l))
    num_section_count = sum(1 for l in lines if RE_NUMBERED_SECTION.match(l))
    return num_section_count > article_count


## Heading Classification

`_classify_heading()` converts a raw heading line into a normalized structural label. It returns four values:

- `level`: the structural role, such as `chapter`, `article`, or `preamble`
- `num`: the heading number or label
- `title`: a readable title for the unit
- `raw_id`: a stable machine-readable ID used internally while parsing headings

This function is where the parser reconciles different constitution styles into one common OHCO model before exporting the simplified corpus columns.


In [4]:
def _classify_heading(line: str, has_chapters: bool, use_numbered: bool) -> tuple[str, str, str, str]:
    """
    Returns (level, num, title, raw_id) where level is one of:
      'chapter', 'article', 'preamble', 'amendment', 'skip'
    """
    # Keep preambles as article-like units so their text stays in the corpus.
    if RE_PREAMBLE.match(line):
        return ('preamble', '0', 'Preamble', 'preamble')

    # Normalize amendment headings into the article layer.
    m = RE_AMENDMENT.match(line)
    if m:
        num = m.group(2)
        title = m.group(3).strip()
        return ('article', f'Amend-{num}', title or f'Amendment {num}', f'amendment_{num}')

    # Treat explicit chapter, part, and title markers as higher-level structure.
    m = RE_CHAPTER.match(line)
    if m:
        num = m.group(2)
        title = m.group(3).strip()
        return ('chapter', num, title or f'Chapter {num}', f'chapter_{num}')

    # Promote Roman-numeral headings only in documents that use chapter-like sections.
    m = RE_ROMAN_SECTION.match(line)
    if m and has_chapters:
        num = m.group(1)
        title = m.group(2).strip()
        return ('chapter', num, title, f'chapter_{num}')

    # Handle standard Article headings, or reinterpret them for US-style documents.
    m = RE_ARTICLE.match(line)
    if m:
        num = m.group(2)
        title = m.group(3).strip()
        if has_chapters:
            return ('article', num, title or f'Article {num}', f'article_{num}')
        else:
            # Treat US-style article headers as chapter-level breaks.
            return ('chapter', num, title or f'Article {num}', f'article_{num}')

    # Map numbered sections to articles when they behave like the main legal unit.
    m = RE_SECTION.match(line)
    if m:
        num = m.group(1)
        title = m.group(2).strip()
        if re.fullmatch(r'\d+[A-Z]?', num):
            return ('article', num, title or f'Section {num}', f'section_{num}')
        return ('chapter', num, title or f'Section {num}', f'section_{num}')

    # Support constitutions where primary sections appear as bare numbered headings.
    m = RE_NUMBERED_SECTION.match(line)
    if m and use_numbered:
        num = m.group(1)
        title = m.group(2).strip()
        return ('article', num, title, f'section_{num}')

    # Leave unmatched headings out of the exported structure.
    return ('skip', '', line, '')


## Parse One Constitution

`parse_file()` is the core parser. It reads one plaintext constitution, extracts country metadata from the first line, detects headings, and walks through the file with a small state machine.

Internally, the parser still tracks larger structural cues such as chapter-like divisions so it can interpret article boundaries correctly. The exported token rows, however, keep only the normalized fields needed for the final corpus: country, article number, clause position, token position, and token text.


In [5]:
def parse_file(filepath: str) -> list[dict]:
    """
    Parse one constitution file into a list of token-level dicts.
    Each dict has: country, article_n,
                   clause_n, token_n, token_str

    Internally the parser still tracks richer heading cues, but the
    exported corpus keeps only the normalized columns used in corpus.csv.
    """
    text = Path(filepath).read_text(encoding='utf-8', errors='replace')
    lines = text.splitlines()

    # Extract the country name from the document header.
    country = 'Unknown'
    if lines:
        m = RE_DOC_HEADER.match(lines[0].strip())
        if m:
            country = m.group(1).strip()

    # Detect which heading style this constitution uses before parsing.
    has_chapters = _has_chapters(lines)
    use_numbered = _uses_numbered_sections(lines)

    # Mark lines as headings when the next non-empty line is "Share".
    heading_lines = set()
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        for j in range(i + 1, len(lines)):
            next_line = lines[j].strip()
            if next_line == 'Share':
                heading_lines.add(i)
                break
            elif next_line:
                break

    # Walk the file once and keep parser state across headings and clauses.
    rows = []
    current_chapter = ''
    current_article_n = 0
    current_clause_n = 0
    token_n = 0
    in_article = False

    def emit_paragraph(para_text: str):
        nonlocal current_article_n, current_clause_n, token_n, in_article
        # Split each paragraph into whitespace-delimited token rows.
        words = para_text.split()
        if not words:
            return
        # Start a new article automatically when preamble text appears before a heading.
        if not in_article:
            current_article_n += 1
            current_clause_n = 0
            in_article = True
        # Treat each emitted paragraph as one clause-like unit.
        current_clause_n += 1
        for word in words:
            word = word.strip()
            if word:
                token_n += 1
                rows.append({
                    'country': country,
                    'article_n': current_article_n,
                    'clause_n': current_clause_n,
                    'token_n': token_n,
                    'token_str': word,
                })

    # Start after the header line and collect content blocks between headings.
    i = 1  # Start after the document header.
    while i < len(lines):
        raw = lines[i]
        stripped = raw.strip()

        # Skip blank lines and Share artifacts.
        if not stripped or RE_SHARE.match(stripped):
            i += 1
            continue

        # Advance parser state when a Share-marked heading is encountered.
        if i in heading_lines:
            level, num, title, raw_id = _classify_heading(stripped, has_chapters, use_numbered)
            if level == 'chapter':
                # Reset article state when the parser enters a higher-level division.
                current_chapter = raw_id
                in_article = False
            elif level in ('article', 'preamble', 'amendment'):
                # Bump the article counter when a new article-like heading appears.
                current_article_n += 1
                current_clause_n = 0
                in_article = True
            # Ignore headings that are detected but not promoted into the exported structure.
            i += 1
            continue

        # Catch chapter-like headings that do not have a following Share line.
        if RE_CHAPTER.match(stripped) or (has_chapters and RE_ROMAN_SECTION.match(stripped)):
            j = i + 1
            while j < len(lines) and not lines[j].strip():
                j += 1
            if j < len(lines):
                next_nonempty = lines[j].strip()
                lvl2, _, _, raw_id2 = _classify_heading(next_nonempty, has_chapters, use_numbered)
                # Accept the heading only when the next real line looks article-like.
                if lvl2 in ('article', 'preamble', 'amendment') or RE_SECTION.match(next_nonempty):
                    current_chapter = raw_id2
                    in_article = False
                    i += 1
                    continue

        # Treat numbered sub-clauses as single clause units before tokenizing.
        if RE_SUBCLAUSE.match(stripped):
            emit_paragraph(stripped)
            i += 1
            continue

        # Merge adjacent content lines into one paragraph until structure changes.
        paragraph = [stripped]
        i += 1
        while i < len(lines):
            nxt = lines[i].strip()
            if not nxt or RE_SHARE.match(nxt) or i in heading_lines:
                break
            paragraph.append(nxt)
            i += 1
        emit_paragraph(' '.join(paragraph))

    return rows


## Add Linguistic Annotation

After the structural parse is complete, the corpus is annotated with token-level linguistic features. This step adds a cleaned `term_str`, a detailed Penn Treebank `pos` tag from NLTK, and a simplified `pos_group` that collapses related tags into broader categories for later analysis.

The cleaned `term_str` is intentionally stricter than `token_str`: it keeps ordinary alphabetic words and standard hyphenated forms, strips enclosing punctuation, lowercases the result, recovers simple leading enumeration patterns such as `3.Discounts` -> `discounts`, and drops noisy artifacts such as pure numbers, single-letter section markers, and punctuation-bound fragments like `d.allow`.


In [6]:
TERM_STRIP_RE = re.compile(r"^[^A-Za-z]+|[^A-Za-z]+$")
LEADING_ENUM_RE = re.compile(r"^\d+(?:[.)-]+)(?=[A-Za-z])")
VALID_TERM_RE = re.compile(r"^[A-Za-z]+(?:-[A-Za-z]+)*$")


def get_pos_group(pos: str) -> str:
    # Collapse detailed POS tags into broader analysis groups.
    if pos.startswith('NN'):
        return 'NN'
    if pos.startswith('VB'):
        return 'VB'
    if pos.startswith('JJ'):
        return 'JJ'
    if pos.startswith('RB'):
        return 'RB'
    if pos in ['IN', 'TO']:
        return 'IN'
    if pos in ['DT', 'PDT', 'WDT']:
        return 'DT'
    if pos in ['PRP', 'PRP$', 'WP', 'WP$']:
        return 'PRON'
    if pos == 'CC':
        return 'CC'
    if pos == 'CD':
        return 'CD'
    if pos in ['$', "''", '(', ')', ',', '--', '.', ':', '``']:
        return 'PUNC'
    return pos


def normalize_term(token: str):
    # Normalize punctuation variants and drop tokens that are not clean terms.
    token = str(token).strip().replace('–', '-').replace('—', '-').replace('−', '-')
    # Trim punctuation from the ends before checking token quality.
    token = TERM_STRIP_RE.sub('', token)
    # Remove leading enumeration like "3." when it is glued to a word.
    token = LEADING_ENUM_RE.sub('', token)
    # Trim again in case enumeration stripping exposed new boundary punctuation.
    token = TERM_STRIP_RE.sub('', token)
    if not token:
        return pd.NA
    # Drop mixed digit tokens because they are usually numbering artifacts here.
    if any(ch.isdigit() for ch in token):
        return pd.NA
    # Drop tokens containing punctuation patterns that are outside the term definition.
    if any(ch in token for ch in ['.', '+', '/', '\\', '=', '*', ':', ';']):
        return pd.NA
    # Keep only alphabetic tokens and standard hyphenated forms.
    if not VALID_TERM_RE.fullmatch(token):
        return pd.NA
    token = token.lower()
    # Exclude one-character remnants after normalization.
    if len(token.replace('-', '')) <= 1:
        return pd.NA
    return token


def add_linguistic_features(df: pd.DataFrame, chunk_size: int = 50000) -> pd.DataFrame:
    # Tag tokens in chunks to avoid large one-shot NLTK calls.
    df = df.copy()
    tokens = df['token_str'].fillna('').astype(str)

    pos_tags = []
    for start in range(0, len(tokens), chunk_size):
        # Batch tagging keeps memory use steadier on the full corpus.
        chunk = tokens.iloc[start:start + chunk_size].tolist()
        pos_tags.extend(tag for _, tag in nltk.pos_tag(chunk))

    # Attach raw POS tags before deriving normalized term features.
    df['pos'] = pos_tags
    df['term_str'] = tokens.apply(normalize_term)
    punc_pos = ['$', "''", '(', ')', ',', '--', '.', ':', '``']
    # Force punctuation-tagged tokens out of the normalized vocabulary space.
    df.loc[df['pos'].isin(punc_pos), 'term_str'] = pd.NA
    # Add a simplified POS grouping for downstream summaries.
    df['pos_group'] = df['pos'].apply(get_pos_group)
    return df


## Build the Full Corpus

`build_corpus()` applies `parse_file()` to every `.txt` file in the input folder and combines all token rows into one pandas DataFrame.

Use the optional `limit` argument when testing. For example, `build_corpus("individual_constitutions", limit=3)` parses only the first three files.


In [7]:
def build_corpus(folder: str, limit: int = None) -> pd.DataFrame:
    """
    Parse all .txt files in folder and return a single normalized DataFrame.
    limit: optionally cap number of files (for testing).
    """
    files = sorted(Path(folder).glob('*.txt'))
    if limit:
        files = files[:limit]

    # Accumulate token rows from each constitution into one table.
    all_rows = []
    for fp in files:
        try:
            rows = parse_file(str(fp))
            all_rows.extend(rows)
            print(f"  {fp.name}: {len(rows):,} tokens")
        except Exception as e:
            print(f"  ERROR {fp.name}: {e}")

    df = pd.DataFrame(all_rows, columns=[
        'country',
        'article_n',
        'clause_n',
        'token_n', 'token_str'
    ])
    return df


## Quick Test on a Few Files

This small test parses only three constitutions so you can confirm the parser works without rebuilding the full 192-document CSV.


In [8]:
test_df = build_corpus('individual_constitutions', limit=3)
print(f'Test tokens: {len(test_df):,}')
test_df.head(10)


  Afghanistan_2004.txt: 9,937 tokens
  Albania_2008.txt: 13,221 tokens
  Algeria_2008.txt: 10,072 tokens
Test tokens: 33,230


,country,article_n,clause_n,token_n,token_str
0,Afghanistan,1,1,1,In
1,Afghanistan,1,1,2,the
2,Afghanistan,1,1,3,name
3,Afghanistan,1,1,4,of
4,Afghanistan,1,1,5,"Allah,"
5,Afghanistan,1,1,6,the
6,Afghanistan,1,1,7,Most
7,Afghanistan,1,1,8,"Beneficent,"
8,Afghanistan,1,1,9,the
9,Afghanistan,1,1,10,Most


## Run the Full Parse, Annotate, and Save CSV

Run this cell when you want to regenerate `corpus.csv`. It parses all constitution files, adds `term_str`, `pos`, and `pos_group`, reports summary statistics, saves the CSV, and displays sample rows from Afghanistan.


In [9]:
CORPUS_DIR = Path('individual_constitutions')
OUTPUT_CSV = Path('corpus.csv')

# Build the full corpus and attach linguistic annotations.
print(f'Parsing constitutions from: {CORPUS_DIR.resolve()}')
df = build_corpus(str(CORPUS_DIR))
print('Adding linguistic features with NLTK...')
df = add_linguistic_features(df)

print(f'\nTotal tokens:       {len(df):,}')
print(f"Unique countries:   {df['country'].nunique()}")
print(f"Avg articles/doc:   {df.groupby('country')['article_n'].max().mean():.1f}")
print(f"Avg clauses/article:{df.groupby(['country','article_n'])['clause_n'].max().mean():.1f}")

df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved to: {OUTPUT_CSV.resolve()}')

print('\nSample rows (Afghanistan):')
df[df['country'].str.startswith('Afghanistan')].head(20)


Parsing constitutions from: C:\Users\garre\school\spring_2026\ds_5001\individual_constitutions
  Afghanistan_2004.txt: 9,937 tokens
  Albania_2008.txt: 13,221 tokens
  Algeria_2008.txt: 10,072 tokens
  Andorra_1993.txt: 8,310 tokens
  Angola_2010.txt: 24,586 tokens
  Antigua_and_Barbuda_1981.txt: 35,033 tokens
  Argentina_1994.txt: 12,571 tokens
  Armenia_2005.txt: 13,312 tokens
  Australia_1985.txt: 12,380 tokens
  Austria_2009.txt: 43,293 tokens
  Azerbaijan_2009.txt: 15,777 tokens
  Bahamas_2002.txt: 36,989 tokens
  Bahrain_2002.txt: 10,470 tokens
  Bangladesh_2011.txt: 24,430 tokens
  Barbados_2007.txt: 33,397 tokens
  Belarus_2004.txt: 13,070 tokens
  Belgium_2012.txt: 15,857 tokens
  Belize_2001.txt: 38,278 tokens
  Benin_1990.txt: 10,889 tokens
  Bhutan_2008.txt: 13,296 tokens
  Bolivia_2009.txt: 38,273 tokens
  Bosnia_Herzegovina_2009.txt: 5,124 tokens
  Botswana_2002.txt: 29,493 tokens
  Brazil_2014.txt: 64,896 tokens
  Brunei_1984.txt: 12,998 tokens
  Bulgaria_2007.txt: 12,01

,country,article_n,clause_n,token_n,token_str,pos,term_str,pos_group
0,Afghanistan,1,1,1,In,IN,in,IN
1,Afghanistan,1,1,2,the,DT,the,DT
2,Afghanistan,1,1,3,name,NN,name,NN
3,Afghanistan,1,1,4,of,IN,of,IN
4,Afghanistan,1,1,5,"Allah,",NNP,allah,NN
5,Afghanistan,1,1,6,the,DT,the,DT
6,Afghanistan,1,1,7,Most,NNP,most,NN
7,Afghanistan,1,1,8,"Beneficent,",NNP,beneficent,NN
8,Afghanistan,1,1,9,the,DT,the,DT
9,Afghanistan,1,1,10,Most,NNP,most,NN


## Notes on Parser Assumptions

This parser is intentionally conservative. It does not try to infer every possible constitutional subdivision; instead, it normalizes the most common structural cues into a practical four-level table.

Important assumptions:

- The first line contains country metadata when available.
- `Share` lines are formatting artifacts and are ignored.
- Paragraph-like text blocks are treated as clauses.
- Larger divisions such as chapters or parts may help the parser interpret structure, but they are not kept as exported corpus columns.
- Tokens are split on whitespace for `token_str`, then `term_str` is cleaned more aggressively: it keeps ordinary alphabetic words and standard hyphenated forms, recovers simple leading enumeration prefixes such as `3.Discounts` -> `discounts`, and drops pure numbers, single-letter markers, and other mixed punctuation artifacts before downstream vocabulary and TFIDF steps.
